# LangWatch Async-Native Experiment Loop with Google ADK

This notebook demonstrates `experiment.aloop` + `experiment.asubmit` — the async sibling of the threading-based `loop` / `submit` pair.

**Why async?** Google ADK's `InMemoryRunner` binds a gRPC channel to the event loop it was created on. Under the threading path each worker runs its coroutine via `asyncio.run(...)`, which spins up a *new* loop per thread and breaks that binding (`"Future attached to a different loop"`). The async-native path keeps every item on the caller's event loop, so singletons and factories that open expensive connections once stay usable across items.

**Requirements**:
```bash
pip install langwatch google-adk openinference-instrumentation-google-adk
```
And set `LANGWATCH_API_KEY` + `GOOGLE_API_KEY` in your environment.

> **Setup:** the ADK runtime is in a separate `adk-examples` group (its pydantic pin conflicts with Chainlit in the main `examples` group). Install with `make install-adk-examples` before running this notebook.


In [1]:
import langwatch
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

langwatch.login()
langwatch.setup(instrumentors=[GoogleADKInstrumentor()])

LangWatch API key is already set, if you want to login again, please call as langwatch.login(relogin=True)
2026-04-17 09:41:23,420 - langwatch.client - INFO - Registering atexit handler to flush tracer provider on exit


/Users/rchaves/Projects/langwatch-async-experiment-parallelism/python-sdk/.venv/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


## Create the agent and a shared runner

The runner is created **once**, on this notebook's event loop. Under the threading path, sharing it across concurrent items would raise `"Future attached to a different loop"`. With `aloop` / `asubmit`, every item runs on this same loop so the runner stays valid.

In [2]:
import os
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types


def get_weather(city: str) -> dict:
    forecasts = {
        "new york": "Sunny, 25C",
        "london": "Cloudy, 18C",
        "tokyo": "Rainy, 22C",
        "berlin": "Windy, 16C",
        "sao paulo": "Humid, 27C",
    }
    city = city.lower().strip()
    if city in forecasts:
        return {"status": "ok", "report": forecasts[city]}
    return {"status": "error", "message": f"no forecast for {city!r}"}


# Use the moving alias so the example keeps running as Google retires
# specific versions. Override with `GEMINI_MODEL=...` for a pinned model.
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-flash-latest")

agent = Agent(
    name="weather_agent",
    model=GEMINI_MODEL,
    description="Replies with a short weather report when asked about a city.",
    instruction="Use the get_weather tool, then answer briefly.",
    tools=[get_weather],
)

runner = InMemoryRunner(agent=agent, app_name="experiment-async-adk")

## Run the experiment with `aloop` + `asubmit`

Each item awaits the shared `runner` concurrently. `concurrency=4` bounds the number of in-flight tasks; the rest queue on an `asyncio.Semaphore`. Sync callables passed to `asubmit` would be offloaded to a worker thread automatically, but here every task is `async`.

In [3]:
from uuid import uuid4

cities = [
    "New York",
    "London",
    "Tokyo",
    "Berlin",
    "Sao Paulo",
    "New York",
    "London",
    "Tokyo",
    "Berlin",
    "Sao Paulo",
]

experiment = langwatch.experiment.init("async-adk-example")

# A fresh prefix per cell execution so session IDs don't collide with earlier
# runs (ADK's in-memory session service raises AlreadyExistsError on duplicates).
RUN_ID = uuid4().hex[:8]


async def ask(city: str, *, item_index: int) -> str:
    session_id = f"run-{RUN_ID}-item-{item_index}"
    await runner.session_service.create_session(
        app_name="experiment-async-adk", user_id="user", session_id=session_id
    )
    # IMPORTANT: drain the whole event stream. Early-returning inside this
    # `async for` leaves the generator to be GC'd later in the event loop's
    # context (not this task's), which tips ADK's OTel wrapper into
    # "Failed to detach context" spam on every iteration.
    response = ""
    async for event in runner.run_async(
        user_id="user",
        session_id=session_id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=f"What is the weather in {city}?")],
        ),
    ):
        if not event.is_final_response():
            continue
        # ADK's is_final_response() can fire for events whose content is
        # None (e.g. state-delta events), so guard every step before
        # indexing parts[0].text.
        content = getattr(event, "content", None)
        parts = getattr(content, "parts", None) or []
        text = getattr(parts[0], "text", None) if parts else None
        if text:
            response = text.strip()
    experiment.log_response(response)
    return response


item_index = 0
async for city in experiment.aloop(cities, concurrency=4):
    experiment.asubmit(ask, city, item_index=item_index)
    item_index += 1

Follow the results at: https://app.langwatch.ai/inbox-narrator/experiments/async-adk-example?runId=platinum-pelican-of-research


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

,entry,output,trace_id,duration_ms,target
index,,,,,
0,New York,The weather in New York is currently sunny and...,82d6d8b2654cd3339a51cba88373864f,2207,Output
1,London,The weather in London is currently cloudy and ...,5a17b0796ee18e6c86cca855d52f95fc,2649,Output
2,Tokyo,The weather in Tokyo is currently rainy with a...,76a3d9b926d31fa2e650afba212aabbc,2539,Output
3,Berlin,The weather in Berlin is currently windy and 1...,6c5b424af2f55fc7d56d013d72fd9ceb,2184,Output
4,Sao Paulo,It's currently humid and 27°C in Sao Paulo.,c310ee5a280f89adfb623b29ce67356c,2513,Output
5,New York,The weather in New York is currently sunny and...,b12cffccc43e8d683ec0d70575e5c1f5,2255,Output
6,London,The weather in London is cloudy and 18°C.,a337de0bf9467774888d2a9202f9b265,2198,Output
7,Tokyo,The weather in Tokyo is rainy with a temperatu...,182c5c4d7068f0761f0a68856d5f2ec8,2246,Output
8,Berlin,The weather in Berlin is currently windy and 1...,a3947bd7cb95d3e50c91b68128da058a,2382,Output



  View details: https://app.langwatch.ai/inbox-narrator/experiments/async-adk-example?runId=platinum-pelican-of-research
  Explore with: `evaluation.results`

